In [ ]:
# SETUP 1 — Install the Python packages used by this notebook.
# sentence-transformers provides BGE, gspread reads/writes Google Sheets, pandas handles tables, and scikit-learn supplies metrics.
!pip install -q sentence-transformers gspread pandas scikit-learn

In [ ]:
# SETUP 2 — Authenticate Colab and connect to Google Sheets.
# The authenticated gspread client is reused to read dataset tabs and later write evaluation results.
from google.colab import auth
auth.authenticate_user()

import google.auth
import gspread
import pandas as pd
import numpy as np

creds, _ = google.auth.default()
gc = gspread.authorize(creds)

In [ ]:
# SETUP 3 — Open the ballot-leakage spreadsheet by URL.
# Listing worksheet titles confirms that TRAIN, VALIDATION, TEST, and summary tabs are available.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1HnVdDmYMZnnEDZANBYWM3RJ0w7WEpXQGZrCxWkyoqMQ/edit"

spreadsheet = gc.open_by_url(SHEET_URL)

print([ws.title for ws in spreadsheet.worksheets()])

['Candidate Dataset', 'Source Ballots', 'Read Me', 'TRAIN', 'VALIDATION', 'TEST', 'SPLIT SUMMARY', 'BGE BASELINE VALIDATION', 'BGE BASELINE METRICS']


In [ ]:
# BASELINE 1 — Load the frozen VALIDATION tab into a pandas DataFrame.
# These 44 rows are used for evaluation and threshold selection, never for updating model weights.
validation_ws = spreadsheet.worksheet("VALIDATION")

validation_df = pd.DataFrame(
    validation_ws.get_all_records()
)

print(validation_df.shape)
validation_df.head()

(61, 15)


,sentence_id,ballot_id,judge_id,actual_ballot,section,sentence,previous_sentence,next_sentence,leakage_strength,leakage_direction,needs_context,annotation_uncertain,uncertainty_reason,data_source,ballot_record_id
0,FA_R1_B2-304_S0016,FA_R1_B2-304,Hailley Yang,CON,Comments to CON,Needs more direct refutation of PRO’s economic...,The logic between economic paralysis and colla...,CON 2 : Good at picking apart CON’s examples; ...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
1,FA_R1_B2-304_S0030,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",Some contentions are not that relevant to the ...,Con2 is also good at pointing out that Pro can...,1,PRO,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
2,FA_R1_B2-304_S0031,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con2 is also good at pointing out that Pro can...,"Con2 also claims that STEMS are wasting money,...",Pro 1 can give more evidence about why exactly...,1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
3,FA_R1_B2-304_S0033,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,Con also points out that proponents fail to gi...,Pro 1 can give more evidence about why exactly...,"Overall speaking, con made more contributions ...",1,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304
4,FA_R1_B2-304_S0034,FA_R1_B2-304,Hailley Yang,CON,Reason for Decision,"Overall speaking, con made more contributions ...",Con also points out that proponents fail to gi...,,2,CON,,false,,additional_verified_csv_2026_09_22,additional_verified_csv_2026_09_22::FA_R1_B2-304


In [ ]:
# BASELINE 2 — Standardize VALIDATION labels and inspect their distributions.
# Numeric strength and normalized direction values prevent type or capitalization errors in later metrics.
validation_df["leakage_strength"] = pd.to_numeric(
    validation_df["leakage_strength"]
)

validation_df["leakage_direction"] = (
    validation_df["leakage_direction"]
    .astype(str)
    .str.upper()
    .str.strip()
)

print(validation_df["leakage_strength"].value_counts())
print()
print(validation_df["leakage_direction"].value_counts())
print()
print(validation_df["section"].value_counts())

leakage_strength
0    28
1    21
2    12
Name: count, dtype: int64

leakage_direction
NEUTRAL    28
CON        19
PRO        14
Name: count, dtype: int64

section
Reason for Decision    34
Comments to CON        14
Comments to PRO        13
Name: count, dtype: int64


In [ ]:
# BASELINE 3 — Load two separate copies of the original BGE checkpoint.
# base_model remains the untouched reference; finetune_model is reserved for Experiment 2 training.
from sentence_transformers import SentenceTransformer

base_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

finetune_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print(
    "Embedding dimension:",
    base_model.get_sentence_embedding_dimension()
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimension: 768


/tmp/ipykernel_568/158451966.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  base_model.get_sentence_embedding_dimension()


In [ ]:
# BASELINE 4 — Define semantic query banks for PRO-favoring and CON-favoring language.
# Multiple paraphrases reduce dependence on any single wording when scoring ballot sentences.
pro_queries = [
    "Pro wins the debate.",
    "The judge votes for Pro.",
    "Pro wins because Con fails to respond to an argument.",
    "Con loses the debate.",
    "Pro wins an important argument or clash."
]

con_queries = [
    "Con wins the debate.",
    "The judge votes for Con.",
    "Con wins because Pro fails to respond to an argument.",
    "Pro loses the debate.",
    "Con wins an important argument or clash."
]

In [ ]:
# BASELINE 5 — Embed the PRO and CON query banks with the untouched model.
# Unit-normalized vectors allow the later matrix dot products to equal cosine similarity.
pro_embeddings = base_model.encode(
    pro_queries,
    normalize_embeddings=True
)

con_embeddings = base_model.encode(
    con_queries,
    normalize_embeddings=True
)

print(pro_embeddings.shape)
print(con_embeddings.shape)

(5, 768)
(5, 768)


In [ ]:
# BASELINE 6 — Embed every VALIDATION sentence with the untouched BGE model.
# Empty values are replaced with blank strings so encoding receives a consistent list of text inputs.
sentences = (
    validation_df["sentence"]
    .fillna("")
    .astype(str)
    .tolist()
)

sentence_embeddings = base_model.encode(
    sentences,
    normalize_embeddings=True
)

print(sentence_embeddings.shape)

(61, 768)


In [ ]:
# BASELINE 7 — Convert embeddings into PRO, CON, direction, and relevance scores.
# direction_score is PRO minus CON; relevance_score keeps the stronger of the two similarities.
pro_similarity = sentence_embeddings @ pro_embeddings.T
con_similarity = sentence_embeddings @ con_embeddings.T

validation_df["pro_score"] = pro_similarity.max(axis=1)
validation_df["con_score"] = con_similarity.max(axis=1)

validation_df["direction_score"] = (
    validation_df["pro_score"]
    - validation_df["con_score"]
)

validation_df["relevance_score"] = np.maximum(
    validation_df["pro_score"],
    validation_df["con_score"]
)

In [ ]:
# BASELINE 8 — Review sentences with the highest semantic relevance scores.
# This is a diagnostic ranking, not a change to labels or the evaluation split.
display_cols = [
    "sentence_id",
    "section",
    "sentence",
    "leakage_strength",
    "leakage_direction",
    "pro_score",
    "con_score",
    "direction_score",
    "relevance_score"
]

validation_df[
    display_cols
].sort_values(
    "relevance_score",
    ascending=False
).head(20)

,sentence_id,section,sentence,leakage_strength,leakage_direction,pro_score,con_score,direction_score,relevance_score
12,FB_R1_B1-209_S0019,Reason for Decision,Pro win this round.,2,PRO,0.791938,0.727637,0.064300,0.791938
50,FA_R1_B1-209_S016,Reason for Decision,"Therefore, this is a CON win.",2,CON,0.743865,0.781392,-0.037527,0.781392
15,FB_R4_3129832024_S0005,Reason for Decision,I vote for PRO.,2,PRO,0.780258,0.720668,0.059590,0.780258
41,FA_R1_B1-209_S003,Comments to PRO,The speaker two rebuttal on CON side being tem...,0,NEUTRAL,0.756340,0.747074,0.009266,0.756340
35,FA_R1_306_S008,Reason for Decision,This a pro Ballot.,2,PRO,0.752982,0.717180,0.035802,0.752982
8,FA_R4_3129832024_S0007,Reason for Decision,I vote for CON.,2,CON,0.743522,0.736148,0.007374,0.743522
5,FA_R2_B1-210_S0010,Reason for Decision,"Both sides did a good job, but when compared, ...",1,CON,0.697526,0.726317,-0.028791,0.726317
32,FA_R1_306_S005,Comments to CON,Good contentions however you need to provide m...,0,NEUTRAL,0.688740,0.726252,-0.037512,0.726252
30,FA_R1_306_S002,Comments to PRO,Good offense and defence in the crossfire to e...,0,NEUTRAL,0.704171,0.724671,-0.020500,0.724671
42,FA_R1_B1-209_S005,Comments to PRO,"Don't use broad terms such as ""uncertain acces...",0,NEUTRAL,0.694563,0.713457,-0.018893,0.713457


In [ ]:
# BASELINE 9 — Print the DataFrame schema after adding model scores.
# This quick check confirms that annotation, metadata, and derived-score columns are all present.
print(validation_df.columns.tolist())

['sentence_id', 'ballot_id', 'judge_id', 'actual_ballot', 'section', 'sentence', 'previous_sentence', 'next_sentence', 'leakage_strength', 'leakage_direction', 'needs_context', 'annotation_uncertain', 'uncertainty_reason', 'data_source', 'ballot_record_id', 'pro_score', 'con_score', 'direction_score', 'relevance_score']


In [ ]:
# BASELINE 10 — Sweep candidate decision thresholds on VALIDATION.
# Positive scores above the threshold predict PRO, negative scores below it predict CON, and the middle band predicts NEUTRAL.
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

def predict_direction(score, threshold):
    if score > threshold:
        return "PRO"
    elif score < -threshold:
        return "CON"
    else:
        return "NEUTRAL"

thresholds = [
    0.00,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10
]

threshold_results = []

for threshold in thresholds:
    predictions = validation_df["direction_score"].apply(
        lambda x: predict_direction(x, threshold)
    )

    accuracy = accuracy_score(
        validation_df["leakage_direction"],
        predictions
    )

    macro_f1 = f1_score(
        validation_df["leakage_direction"],
        predictions,
        average="macro"
    )

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values(
    "macro_f1",
    ascending=False
)

,threshold,accuracy,macro_f1
2,0.020,0.540984,0.481313
3,0.030,0.557377,0.474224
1,0.010,0.475410,0.466989
4,0.040,0.508197,0.334701
0,0.000,0.377049,0.324973
5,0.050,0.491803,0.297893
6,0.075,0.459016,0.209738
7,0.100,0.459016,0.209738


In [ ]:
# BASELINE 11 — Select the threshold with the best VALIDATION Macro-F1.
# Macro-F1 weights all three classes equally and is therefore less dominated by the larger NEUTRAL class.
best_row = threshold_results_df.loc[
    threshold_results_df["macro_f1"].idxmax()
]

BEST_THRESHOLD = float(best_row["threshold"])

print("Best threshold:", BEST_THRESHOLD)
print("Validation Macro-F1:", best_row["macro_f1"])
print("Validation Accuracy:", best_row["accuracy"])

Best threshold: 0.02
Validation Macro-F1: 0.48131331850753645
Validation Accuracy: 0.5409836065573771


In [ ]:
# BASELINE 12 — Apply the selected threshold to every VALIDATION direction score.
# The resulting predicted_direction column is used by all following baseline metrics.
validation_df["predicted_direction"] = (
    validation_df["direction_score"]
    .apply(
        lambda x: predict_direction(
            x,
            BEST_THRESHOLD
        )
    )
)

In [ ]:
# BASELINE 13 — Preview true labels, scores, and predictions side by side.
# This human-readable sample helps verify that the threshold logic behaves as intended.
validation_df[
    [
        "sentence",
        "leakage_strength",
        "leakage_direction",
        "direction_score",
        "predicted_direction"
    ]
].head(20)

,sentence,leakage_strength,leakage_direction,direction_score,predicted_direction
0,Needs more direct refutation of PRO’s economic...,1,PRO,0.008023,NEUTRAL
1,"Con2 also claims that STEMS are wasting money,...",1,PRO,-0.000523,NEUTRAL
2,Con2 is also good at pointing out that Pro can...,1,CON,-0.001340,NEUTRAL
3,Con also points out that proponents fail to gi...,1,CON,0.011090,NEUTRAL
4,"Overall speaking, con made more contributions ...",2,CON,-0.017609,NEUTRAL
5,"Both sides did a good job, but when compared, ...",1,CON,-0.028791,CON
6,"Therefore, I voted for the con side.",2,CON,-0.038615,CON
7,Your strongest move was asking whether PRO cou...,1,CON,0.012720,NEUTRAL
8,I vote for CON.,2,CON,0.007374,NEUTRAL
9,PRO explained well why childcare can reduce pr...,1,CON,0.021694,PRO


In [ ]:
# BASELINE 14 — Print class-level precision, recall, and F1 on all VALIDATION sentences.
# zero_division=0 prevents undefined metrics from raising warnings when a class receives no predictions.
from sklearn.metrics import classification_report, confusion_matrix

print("=== ALL VALIDATION SENTENCES ===")

print(
    classification_report(
        validation_df["leakage_direction"],
        validation_df["predicted_direction"],
        labels=["PRO", "CON", "NEUTRAL"],
        zero_division=0
    )
)

=== ALL VALIDATION SENTENCES ===
              precision    recall  f1-score   support

         PRO       0.80      0.29      0.42        14
         CON       0.46      0.32      0.38        19
     NEUTRAL       0.53      0.82      0.65        28

    accuracy                           0.54        61
   macro avg       0.60      0.47      0.48        61
weighted avg       0.57      0.54      0.51        61



In [ ]:
# BASELINE 15 — Build a three-class confusion matrix.
# Fixed row and column labels make the direction of each PRO, CON, or NEUTRAL error explicit.
cm = confusion_matrix(
    validation_df["leakage_direction"],
    validation_df["predicted_direction"],
    labels=["PRO", "CON", "NEUTRAL"]
)

cm_df = pd.DataFrame(
    cm,
    index=["True PRO", "True CON", "True NEUTRAL"],
    columns=["Pred PRO", "Pred CON", "Pred NEUTRAL"]
)

cm_df

,Pred PRO,Pred CON,Pred NEUTRAL
True PRO,4,2,8
True CON,1,6,12
True NEUTRAL,0,5,23


In [ ]:
# BASELINE 16 — Evaluate only sentences manually annotated as leaking.
# Filtering leakage_strength > 0 removes true NEUTRAL rows and focuses the report on directional leakage.
leaking_df = validation_df[
    validation_df["leakage_strength"] > 0
].copy()

print("Number of leakage sentences:", len(leaking_df))

print(
    classification_report(
        leaking_df["leakage_direction"],
        leaking_df["predicted_direction"],
        labels=["PRO", "CON"],
        zero_division=0
    )
)

Number of leakage sentences: 33
              precision    recall  f1-score   support

         PRO       0.80      0.29      0.42        14
         CON       0.75      0.32      0.44        19

   micro avg       0.77      0.30      0.43        33
   macro avg       0.78      0.30      0.43        33
weighted avg       0.77      0.30      0.43        33



In [ ]:
# BASELINE 17 — Calculate direction accuracy within the leakage-only subset.
# A prediction counts as correct only when it identifies the annotated leaking side.
leaking_direction_accuracy = accuracy_score(
    leaking_df["leakage_direction"],
    leaking_df["predicted_direction"]
)

print(
    "Direction accuracy among leakage sentences:",
    round(leaking_direction_accuracy, 3)
)

Direction accuracy among leakage sentences: 0.303


In [ ]:
# BASELINE 18 — Measure direction accuracy on the strongest leakage examples.
# leakage_strength == 2 isolates sentences that most explicitly reveal a ballot direction.
strong_df = validation_df[
    validation_df["leakage_strength"] == 2
].copy()

print("Strength-2 examples:", len(strong_df))

if len(strong_df) > 0:
    strong_accuracy = accuracy_score(
        strong_df["leakage_direction"],
        strong_df["predicted_direction"]
    )

    print(
        "Strength-2 direction accuracy:",
        round(strong_accuracy, 3)
    )

Strength-2 examples: 12
Strength-2 direction accuracy: 0.5


In [ ]:
# BASELINE 19 — Define ballot-level Precision@k.
# For each ballot, the function ranks sentences by relevance and measures how many of the top k are truly leaking.
def precision_at_k(group, k=3):
    ranked = group.sort_values(
        "relevance_score",
        ascending=False
    )

    top_k = ranked.head(
        min(k, len(ranked))
    )

    return (
        top_k["leakage_strength"] > 0
    ).mean()

In [ ]:
# BASELINE 20 — Average Precision@1, @3, and @5 across ballots.
# Grouping by ballot_id prevents ballots with more selected sentences from dominating the ranking metric.
precision_results = []

for k in [1, 3, 5]:

    ballot_scores = (
        validation_df
        .groupby("ballot_id")
        .apply(
            lambda g: precision_at_k(g, k),
            include_groups=False
        )
    )

    mean_score = ballot_scores.mean()

    precision_results.append({
        "metric": f"Precision@{k}",
        "score": mean_score
    })

    print(
        f"Precision@{k}:",
        round(mean_score, 3)
    )

Precision@1: 1.0
Precision@3: 0.708
Precision@5: 0.775


In [ ]:
# BASELINE 21 — Define ballot-level normalized discounted cumulative gain.
# nDCG rewards ranking stronger leakage labels near the top while accounting for their exact positions.
from sklearn.metrics import ndcg_score
import numpy as np

def ballot_ndcg(group, k=5):

    if len(group) < 2:
        return np.nan

    true_relevance = np.array([
        group["leakage_strength"].astype(float).values
    ])

    predicted_scores = np.array([
        group["relevance_score"].astype(float).values
    ])

    return ndcg_score(
        true_relevance,
        predicted_scores,
        k=min(k, len(group))
    )

In [ ]:
# BASELINE 22 — Average nDCG@3 and nDCG@5 across eligible ballots.
# Ballots with fewer than two sentences return NaN and are excluded from the mean.
ndcg_results = []

for k in [3, 5]:

    ballot_scores = (
        validation_df
        .groupby("ballot_id")
        .apply(
            lambda g: ballot_ndcg(g, k),
            include_groups=False
        )
        .dropna()
    )

    mean_score = ballot_scores.mean()

    ndcg_results.append({
        "metric": f"nDCG@{k}",
        "score": mean_score
    })

    print(
        f"nDCG@{k}:",
        round(mean_score, 3)
    )

nDCG@3: 0.792
nDCG@5: 0.831


In [ ]:
# BASELINE 23 — Surface likely false-positive candidates.
# These are non-leaking sentences with high relevance scores that the embedding method may incorrectly prioritize.
false_positive_candidates = (
    validation_df[
        validation_df["leakage_strength"] == 0
    ]
    .sort_values(
        "relevance_score",
        ascending=False
    )
)

false_positive_candidates[
    [
        "sentence_id",
        "section",
        "sentence",
        "leakage_direction",
        "pro_score",
        "con_score",
        "direction_score",
        "relevance_score",
        "predicted_direction"
    ]
].head(15)

,sentence_id,section,sentence,leakage_direction,pro_score,con_score,direction_score,relevance_score,predicted_direction
41,FA_R1_B1-209_S003,Comments to PRO,The speaker two rebuttal on CON side being tem...,NEUTRAL,0.756340,0.747074,0.009266,0.756340,NEUTRAL
32,FA_R1_306_S005,Comments to CON,Good contentions however you need to provide m...,NEUTRAL,0.688740,0.726252,-0.037512,0.726252,CON
30,FA_R1_306_S002,Comments to PRO,Good offense and defence in the crossfire to e...,NEUTRAL,0.704171,0.724671,-0.020500,0.724671,CON
42,FA_R1_B1-209_S005,Comments to PRO,"Don't use broad terms such as ""uncertain acces...",NEUTRAL,0.694563,0.713457,-0.018893,0.713457,NEUTRAL
45,FA_R1_B1-209_S009,Comments to CON,very good crossfire response to the opponents ...,NEUTRAL,0.703650,0.696444,0.007206,0.703650,NEUTRAL
53,FB_R1_B1-209_S004,Comments to PRO,The control was good on the opponents from you...,NEUTRAL,0.674112,0.702698,-0.028585,0.702698,CON
22,FA_R1_301_S014,Comments to CON,Offered rebuttals addressing Pro’s assumptions...,NEUTRAL,0.689176,0.671693,0.017484,0.689176,NEUTRAL
52,FB_R1_B1-209_S003,Comments to PRO,Good when you looked at that in the rebuttal.,NEUTRAL,0.687069,0.682845,0.004224,0.687069,NEUTRAL
34,FA_R1_306_S007,Comments to CON,"Also, summarize your contentions in the summar...",NEUTRAL,0.669377,0.671070,-0.001693,0.671070,NEUTRAL
23,FA_R1_301_S016,Comments to CON,Some arguments could sound too broad—clarity a...,NEUTRAL,0.644728,0.665642,-0.020913,0.665642,CON


In [ ]:
# BASELINE 24 — Surface likely missed-leakage candidates.
# These are annotated leakage sentences with unusually low relevance scores and are useful for qualitative error review.
missed_leakage = (
    validation_df[
        validation_df["leakage_strength"] > 0
    ]
    .sort_values(
        "relevance_score",
        ascending=True
    )
)

missed_leakage[
    [
        "sentence_id",
        "section",
        "sentence",
        "leakage_strength",
        "leakage_direction",
        "pro_score",
        "con_score",
        "direction_score",
        "relevance_score",
        "predicted_direction"
    ]
].head(15)

,sentence_id,section,sentence,leakage_strength,leakage_direction,pro_score,con_score,direction_score,relevance_score,predicted_direction
59,FB_R1_B1-209_S013,Reason for Decision,The PRO side proved that the flexibility is on...,1,PRO,0.490318,0.484829,0.005488,0.490318,NEUTRAL
9,FA_R4_3129832024_S0008,Reason for Decision,PRO explained well why childcare can reduce pr...,1,CON,0.544751,0.523057,0.021694,0.544751,PRO
25,FA_R1_301_S020,Reason for Decision,"However, the Con successfully argued that thes...",1,CON,0.585542,0.591468,-0.005925,0.591468,NEUTRAL
48,FA_R1_B1-209_S014,Reason for Decision,"They also controlled the entire debate, clearl...",2,CON,0.590131,0.595225,-0.005094,0.595225,NEUTRAL
1,FA_R1_B2-304_S0030,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",1,PRO,0.600039,0.600563,-0.000523,0.600563,NEUTRAL
39,FA_R1_306_S013,Reason for Decision,The pro side countered this and stated how thi...,1,PRO,0.617112,0.599637,0.017474,0.617112,NEUTRAL
49,FA_R1_B1-209_S015,Reason for Decision,The CON side clearly proved how the accessibil...,1,CON,0.587180,0.620246,-0.033066,0.620246,CON
13,FB_R1_B1-209_S0025,Reason for Decision,Pro was able to prove that there is already a ...,1,PRO,0.620276,0.576800,0.043475,0.620276,PRO
27,FA_R1_301_S023,Reason for Decision,Regarding Pro’s proposed solution—using AI to ...,1,CON,0.622454,0.622884,-0.000430,0.622884,NEUTRAL
60,FB_R1_B1-209_S014,Reason for Decision,The CON side could not prove how the currect s...,1,PRO,0.621998,0.628403,-0.006405,0.628403,NEUTRAL


In [ ]:
# BASELINE 25 — Inspect leaking sentences assigned the wrong direction.
# Sorting by leakage strength places the most consequential PRO/CON reversals first.
wrong_direction = validation_df[
    (
        validation_df["leakage_strength"] > 0
    )
    &
    (
        validation_df["leakage_direction"]
        !=
        validation_df["predicted_direction"]
    )
].copy()

wrong_direction[
    [
        "sentence_id",
        "section",
        "sentence",
        "leakage_strength",
        "leakage_direction",
        "predicted_direction",
        "pro_score",
        "con_score",
        "direction_score"
    ]
].sort_values(
    "leakage_strength",
    ascending=False
)

,sentence_id,section,sentence,leakage_strength,leakage_direction,predicted_direction,pro_score,con_score,direction_score
4,FA_R1_B2-304_S0034,Reason for Decision,"Overall speaking, con made more contributions ...",2,CON,NEUTRAL,0.689932,0.707542,-0.017609
8,FA_R4_3129832024_S0007,Reason for Decision,I vote for CON.,2,CON,NEUTRAL,0.743522,0.736148,0.007374
10,FA_R4_3129832024_S0010,Reason for Decision,"Most importantly, CON directly challenged PRO’...",2,CON,NEUTRAL,0.658137,0.643990,0.014147
28,FA_R1_301_S024,Reason for Decision,"Con won by providing clearer causal logic, by ...",2,CON,NEUTRAL,0.647605,0.655191,-0.007587
48,FA_R1_B1-209_S014,Reason for Decision,"They also controlled the entire debate, clearl...",2,CON,NEUTRAL,0.590131,0.595225,-0.005094
17,FB_R4_3129832024_S0007,Reason for Decision,"CON’s Final Focus on cost, choice, and speed w...",2,PRO,NEUTRAL,0.672675,0.664224,0.008451
0,FA_R1_B2-304_S0016,Comments to CON,Needs more direct refutation of PRO’s economic...,1,PRO,NEUTRAL,0.668294,0.660271,0.008023
7,FA_R4_3129832024_S0005,Comments to CON,Your strongest move was asking whether PRO cou...,1,CON,NEUTRAL,0.649849,0.637129,0.012720
3,FA_R1_B2-304_S0033,Reason for Decision,Con also points out that proponents fail to gi...,1,CON,NEUTRAL,0.674736,0.663646,0.011090
1,FA_R1_B2-304_S0030,Reason for Decision,"Con2 also claims that STEMS are wasting money,...",1,PRO,NEUTRAL,0.600039,0.600563,-0.000523


In [ ]:
# BASELINE 26 — Compare performance across ballot sections.
# Accuracy, Macro-F1, and mean relevance are calculated separately for comments to each side and reasons for decision.
section_results = []

for section, subset in validation_df.groupby("section"):

    accuracy = accuracy_score(
        subset["leakage_direction"],
        subset["predicted_direction"]
    )

    macro_f1 = f1_score(
        subset["leakage_direction"],
        subset["predicted_direction"],
        average="macro"
    )

    section_results.append({
        "section": section,
        "n": len(subset),
        "direction_accuracy": accuracy,
        "macro_f1": macro_f1,
        "mean_relevance_score": subset["relevance_score"].mean()
    })

section_results_df = pd.DataFrame(section_results)

section_results_df

,section,n,direction_accuracy,macro_f1,mean_relevance_score
0,Comments to CON,14,0.642857,0.272727,0.644850
1,Comments to PRO,13,0.769231,0.628571,0.650768
2,Reason for Decision,34,0.411765,0.420838,0.652324


In [ ]:
# BASELINE 27 — Summarize relevance scores by manual leakage strength.
# The grouped statistics show whether stronger leakage annotations tend to receive higher semantic relevance.
validation_df.groupby(
    "leakage_strength"
)["relevance_score"].agg(
    ["count", "mean", "median", "std"]
)

,count,mean,median,std
leakage_strength,,,,
0,28,0.631884,0.618778,0.064417
1,21,0.641492,0.655043,0.054943
2,12,0.708568,0.704942,0.062055


In [ ]:
# BASELINE 28 — Assemble the headline baseline and ranking metrics.
# Rounding occurs only for display; the underlying calculations use the unrounded values.
overall_accuracy = accuracy_score(
    validation_df["leakage_direction"],
    validation_df["predicted_direction"]
)

overall_macro_f1 = f1_score(
    validation_df["leakage_direction"],
    validation_df["predicted_direction"],
    average="macro"
)

metrics_summary = [
    {
        "metric": "Best direction threshold",
        "score": BEST_THRESHOLD
    },
    {
        "metric": "Direction Accuracy",
        "score": overall_accuracy
    },
    {
        "metric": "Direction Macro-F1",
        "score": overall_macro_f1
    },
    {
        "metric": "Leakage-only Direction Accuracy",
        "score": leaking_direction_accuracy
    }
]

metrics_summary += precision_results
metrics_summary += ndcg_results

metrics_summary_df = pd.DataFrame(
    metrics_summary
)

metrics_summary_df["score"] = (
    metrics_summary_df["score"]
    .astype(float)
    .round(4)
)

metrics_summary_df

,metric,score
0,Best direction threshold,0.0200
1,Direction Accuracy,0.5410
2,Direction Macro-F1,0.4813
3,Leakage-only Direction Accuracy,0.3030
4,Precision@1,1.0000
5,Precision@3,0.7083
6,Precision@5,0.7750
7,nDCG@3,0.7917
8,nDCG@5,0.8311


In [ ]:
# EXPORT 1 — Write sentence-level baseline results back to Google Sheets.
# Re-running this cell replaces the existing BGE BASELINE VALIDATION tab, then uploads the current validation table.
results_tab_name = "BGE BASELINE VALIDATION"

try:
    old_ws = spreadsheet.worksheet(
        results_tab_name
    )
    spreadsheet.del_worksheet(old_ws)
except gspread.WorksheetNotFound:
    pass

results_ws = spreadsheet.add_worksheet(
    title=results_tab_name,
    rows=len(validation_df) + 5,
    cols=len(validation_df.columns) + 2
)

output_df = validation_df.fillna("").copy()

results_ws.update(
    range_name="A1",
    values=[
        output_df.columns.tolist()
    ] + output_df.astype(str).values.tolist()
)

print("Saved:", results_tab_name)

Saved: BGE BASELINE VALIDATION


In [ ]:
# EXPORT 2 — Write the baseline metric summary to a dedicated Google Sheets tab.
# Re-running replaces the existing metrics tab so the spreadsheet reflects the current notebook results.
metrics_tab_name = "BGE BASELINE METRICS"

try:
    old_ws = spreadsheet.worksheet(
        metrics_tab_name
    )
    spreadsheet.del_worksheet(old_ws)
except gspread.WorksheetNotFound:
    pass

metrics_ws = spreadsheet.add_worksheet(
    title=metrics_tab_name,
    rows=100,
    cols=10
)

metrics_ws.update(
    range_name="A1",
    values=[
        ["ORIGINAL BGE VALIDATION BASELINE"],
        [],
        ["Metric", "Score"]
    ] + metrics_summary_df.astype(str).values.tolist()
)

print("Saved:", metrics_tab_name)

Saved: BGE BASELINE METRICS


In [ ]:
# EXPORT 3 — Append the section-level analysis below the headline metrics.
# start_row leaves spacing between the overall summary and the per-section table.
start_row = len(metrics_summary_df) + 7

metrics_ws.update(
    range_name=f"A{start_row}",
    values=[
        ["SECTION ANALYSIS"],
        [],
        section_results_df.columns.tolist()
    ] + section_results_df.astype(str).values.tolist()
)

{'spreadsheetId': '1HnVdDmYMZnnEDZANBYWM3RJ0w7WEpXQGZrCxWkyoqMQ',
 'updatedRange': "'BGE BASELINE METRICS'!A16:E21",
 'updatedRows': 5,
 'updatedColumns': 5,
 'updatedCells': 21}